In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

ROOT = Path(r"C:/Users/Carl/Desktop/RA Data/Covariate balance")

FILES = {
    "csb": ROOT / "CSB_panel_v2.xlsx",
    "control": ROOT / "Village_candidates_panel_v2.xlsx",
    "treatment_waves": ROOT / "CSB treatment waves.xlsx",
    "distance": ROOT / "distance_major_town.xlsx",
    "elevation": ROOT / "elevation.xlsx",
    "lccs": ROOT / "LCCS.xlsx",
    "poverty": ROOT / "poverty_2010.xlsx",
    "temp": ROOT / "Temp.xlsx",
}

WAVE_YEARS = {
    1: {"wave_start_year": 2009, "wave_end_year": 2010, "wave_label": "2009-2010"},
    2: {"wave_start_year": 2010, "wave_end_year": 2011, "wave_label": "2010-2011"},
    3: {"wave_start_year": 2011, "wave_end_year": 2012, "wave_label": "2011-2012"},
    4: {"wave_start_year": 2013, "wave_end_year": 2014, "wave_label": "2013-2014"},
    5: {"wave_start_year": 2015, "wave_end_year": 2016, "wave_label": "2015-2016"},
    7: {"wave_start_year": 2018, "wave_end_year": 2019, "wave_label": "2018-2019"},
    8: {"wave_start_year": 2019, "wave_end_year": 2020, "wave_label": "2019-2020"},
}

def clean_key(x):
    if pd.isna(x):
        return pd.NA
    return " ".join(str(x).strip().upper().split())

def num(s):
    return pd.to_numeric(s, errors="coerce")

def load_excel(path, sheet=0):
    df = pd.read_excel(path, sheet_name=sheet, dtype=str)
    df.columns = df.columns.str.strip()
    return df

def weighted_mean(x, w):
    x = num(x)
    w = num(w)
    ok = x.notna() & w.notna() & w.gt(0)
    if ok.any():
        return np.average(x[ok], weights=w[ok])
    return x.mean()

def extract_first_wave(x):
    if pd.isna(x):
        return pd.NA
    m = re.search(r"Wave\s*(\d+)", str(x), flags=re.IGNORECASE)
    return int(m.group(1)) if m else pd.NA

def merge_cov(df, cov, on, cols):
    before = len(df)
    cov = cov[[on] + cols].drop_duplicates(on)
    out = df.merge(cov, on=on, how="left", validate="m:1")
    if len(out) != before:
        raise ValueError("Row count changed during merge.")
    return out

In [3]:
# treatment timing

treat_waves = load_excel(FILES["treatment_waves"])

treat_waves["site_key"] = treat_waves["CSB"].map(clean_key)
treat_waves["treat_year"] = num(treat_waves["Established"])
treat_waves["latest_baseline_wave"] = (
    treat_waves["Latest wave compatible"]
    .map(extract_first_wave)
    .astype("Int64")
)

treat_waves_map = (
    treat_waves[
        ["site_key", "treat_year", "latest_baseline_wave", "Latest wave compatible"]
    ]
    .drop_duplicates("site_key")
)

display(treat_waves_map)

,site_key,treat_year,latest_baseline_wave,Latest wave compatible
0,KIZIMBA CSB,2010,1,Wave 1 - borderline wave 2
1,NAWANDALA CSB,2013,4,Wave 4
2,NAKASEKE CSB,2014,4,Wave 4
3,RUBAYA CSB,2014,4,Wave 4
4,JOY M.CSB,2016,5,Wave 5
5,MIC PA RWOTH,2016,5,Wave 5
6,SOROTI CSB,2017,5,Wave 5
7,ONGAKO CSB,2017,5,Wave 5
8,NEM ANYIM CSB,2018,7,Wave 7
9,HOIMA CSB,2018,7,Wave 7


In [4]:

csb = load_excel(FILES["csb"])
ctrl = load_excel(FILES["control"])

csb["site_id"] = "CSB_" + csb["Seed Bank"].map(clean_key)
csb["site_name"] = csb["Seed Bank"]
csb["site_key"] = csb["Seed Bank"].map(clean_key)
csb["treated"] = 1

ctrl["site_id"] = "CTRL_" + ctrl["index"].astype(str)
ctrl["site_name"] = ctrl["name_11k"]
ctrl["site_key"] = ctrl["index"].astype(str).str.strip()
ctrl["treated"] = 0

for df in [csb, ctrl]:
    df["Wave"] = num(df["Wave"]).astype("Int64")
    df["wave_start_year"] = df["Wave"].map({k: v["wave_start_year"] for k, v in WAVE_YEARS.items()})
    df["wave_end_year"] = df["Wave"].map({k: v["wave_end_year"] for k, v in WAVE_YEARS.items()})
    df["wave_label"] = df["Wave"].map({k: v["wave_label"] for k, v in WAVE_YEARS.items()})
    df["n_HHID_wave"] = num(df["n_HHID_wave"])

csb = csb.merge(treat_waves_map, on="site_key", how="left", validate="m:1")

ctrl["treat_year"] = pd.NA
ctrl["latest_baseline_wave"] = pd.NA
ctrl["Latest wave compatible"] = pd.NA



In [6]:
# rename core variables

rename_common = {
    "avg_subsistence": "avg_subsistence_base",
    "AGSEC10_ANY": "ag_extension_base",
    "AGSEC10_Service_avg": "ag_extension_base",
    "H18Q1A": "road_type_A_base",
    "H18Q1B": "road_type_B_base",
    "H18Q1C": "road_type_C_base",
    "H18Q1D": "road_type_D_base",
    "H18Q4A": "time_to_road_A_base",
    "H18Q4B": "time_to_road_B_base",
    "H18Q4C": "time_to_road_C_base",
    "H18Q4D": "time_to_road_D_base",
    "A3Q4_1": "organic_fert_s1_base",
    "A3Q4_2": "organic_fert_s2_base",
    "A3Q14_1": "inorganic_fert_s1_base",
    "A3Q14_2": "inorganic_fert_s2_base",
    "A3Q26_1": "pesticide_use_s1_base",
    "A3Q26_2": "pesticide_use_s2_base",
    "A3Q39_1": "days_plot_s1_base",
    "A3Q39_2": "days_plot_s2_base",
    "A3Q41_1": "hired_labor_s1_base",
    "A3Q41_2": "hired_labor_s2_base",
    "A3Q43_1": "labor_cash_s1_base",
    "A3Q43_2": "labor_cash_s2_base",
    "share_hh_working_1": "share_hh_working_s1_base",
    "share_hh_working_2": "share_hh_working_s2_base",
    "Mean_mm_season_1": "rain_mm_s1_2000_2010",
    "Mean_mm_season_2": "rain_mm_s2_2000_2010",
    "density_2010": "popdens_2010",
    "Trend": "popdens_trend_2000_2010",
    "Densitytrend_2000_2020": "popdens_trend_2000_2010",
}

rename_common.update({
    "H12Q01_avg": "non_agriculture_enterprise_avg_base",
    "H10Q1_avg": "electricity_access_avg_base",
})


csb = csb.rename(columns={k: v for k, v in rename_common.items() if k in csb.columns})
ctrl = ctrl.rename(columns={k: v for k, v in rename_common.items() if k in ctrl.columns})

In [7]:
# Merge GIS / satellite covariates

dist_ctrl = load_excel(FILES["distance"], "Sheet1")
dist_ctrl["site_key"] = dist_ctrl["index"].astype(str).str.strip()
dist_ctrl["dist_major_town_km"] = num(dist_ctrl["HubDist"]) / 1000
ctrl = merge_cov(ctrl, dist_ctrl, "site_key", ["dist_major_town_km"])

dist_csb = load_excel(FILES["distance"], "Sheet2")
dist_csb["site_key"] = dist_csb["CSB"].map(clean_key)
dist_csb["dist_major_town_km"] = num(dist_csb["dist_town"]) / 1000
csb = merge_cov(csb, dist_csb, "site_key", ["dist_major_town_km"])

elev_ctrl = load_excel(FILES["elevation"], "Candidates")
elev_ctrl["site_key"] = elev_ctrl["index"].astype(str).str.strip()
elev_ctrl = elev_ctrl.rename(columns={"_mean": "elev_mean_30km", "_stdev": "elev_sd_30km"})
ctrl = merge_cov(ctrl, elev_ctrl, "site_key", ["elev_mean_30km", "elev_sd_30km"])

elev_csb = load_excel(FILES["elevation"], "CSB")
elev_csb["site_key"] = elev_csb["Seed Bank"].map(clean_key)
elev_csb = elev_csb.rename(columns={"_mean": "elev_mean_30km", "_stdev": "elev_sd_30km"})
csb = merge_cov(csb, elev_csb, "site_key", ["elev_mean_30km", "elev_sd_30km"])

lccs_ctrl = load_excel(FILES["lccs"], "Villages ")
lccs_ctrl["site_key"] = lccs_ctrl["index"].astype(str).str.strip()
lccs_ctrl = lccs_ctrl.rename(columns={
    "lc10_2010": "cropland_10_share_2010",
    "lc20_2010": "cropland_20_share_2010",
    "lc30_2010": "cropland_30_share_2010",
    "lc40_2010": "cropland_40_share_2010",
})
ctrl = merge_cov(ctrl, lccs_ctrl, "site_key", [
    "cropland_10_share_2010", "cropland_20_share_2010",
    "cropland_30_share_2010", "cropland_40_share_2010"
])

lccs_csb = load_excel(FILES["lccs"], "CSB")
lccs_csb["site_key"] = lccs_csb["Seed Bank"].map(clean_key)
lccs_csb = lccs_csb.rename(columns={
    "lc10_2010": "cropland_10_share_2010",
    "lc20_2010": "cropland_20_share_2010",
    "lc30_2010": "cropland_30_share_2010",
    "lc40_2010": "cropland_40_share_2010",
})
csb = merge_cov(csb, lccs_csb, "site_key", [
    "cropland_10_share_2010", "cropland_20_share_2010",
    "cropland_30_share_2010", "cropland_40_share_2010"
])

pov_ctrl = load_excel(FILES["poverty"], "Sheet1")
pov_ctrl["site_key"] = pov_ctrl["index"].astype(str).str.strip()
pov_ctrl = pov_ctrl.rename(columns={
    "povert_2010_mean": "poverty_mean_2010",
    "poverty_stdev": "poverty_sd_2010",
})
ctrl = merge_cov(ctrl, pov_ctrl, "site_key", ["poverty_mean_2010", "poverty_sd_2010"])

pov_csb = load_excel(FILES["poverty"], "Sheet2")
pov_csb["site_key"] = pov_csb["Seed Bank"].map(clean_key)
pov_csb = pov_csb.rename(columns={"_mean": "poverty_mean_2010", "_stdev": "poverty_sd_2010"})
csb = merge_cov(csb, pov_csb, "site_key", ["poverty_mean_2010", "poverty_sd_2010"])

temp_ctrl = load_excel(FILES["temp"], "Sheet2")
temp_ctrl["site_key"] = temp_ctrl["index"].astype(str).str.strip()
temp_ctrl = temp_ctrl.rename(columns={
    "Mean_temp_s1_2000-2010": "temp_s1_2000_2010",
    "Mean_temp_s2_2000-2010": "temp_s2_2000_2010",
})
ctrl = merge_cov(ctrl, temp_ctrl, "site_key", ["temp_s1_2000_2010", "temp_s2_2000_2010"])

temp_csb = load_excel(FILES["temp"], "CSB")
temp_csb["site_key"] = temp_csb["CSB"].map(clean_key)
temp_csb = temp_csb.rename(columns={
    "Mean_temp_s1_2000-2010": "temp_s1_2000_2010",
    "Mean_temp_s2_2000-2010": "temp_s2_2000_2010",
})
csb = merge_cov(csb, temp_csb, "site_key", ["temp_s1_2000_2010", "temp_s2_2000_2010"])



In [8]:
expected_static_covars = [
    "rain_mm_s1_2000_2010",
    "rain_mm_s2_2000_2010",
    "temp_s1_2000_2010",
    "temp_s2_2000_2010",
    "popdens_2010",
    "popdens_trend_2000_2010",
    "dist_major_town_km",
    "elev_mean_30km",
    "elev_sd_30km",
    "cropland_10_share_2010",
    "cropland_20_share_2010",
    "cropland_30_share_2010",
    "cropland_40_share_2010",
    "poverty_mean_2010",
    "poverty_sd_2010",
]

print("Missing from CSB:")
print([c for c in expected_static_covars if c not in csb.columns])

print("\nMissing from controls:")
print([c for c in expected_static_covars if c not in ctrl.columns])

static_covars = expected_static_covars.copy()

Missing from CSB:
[]

Missing from controls:
[]


In [9]:
# Collapse to one baseline row per site

baseline_vars = [
    "share_hh_working_s1_base", "share_hh_working_s2_base",
    "avg_subsistence_base", "ag_extension_base",
    "road_type_A_base", "road_type_B_base", "road_type_C_base", "road_type_D_base",
    "time_to_road_A_base", "time_to_road_B_base", "time_to_road_C_base", "time_to_road_D_base",
    "pesticide_use_s1_base", "pesticide_use_s2_base",
    "organic_fert_s1_base", "organic_fert_s2_base",
    "inorganic_fert_s1_base", "inorganic_fert_s2_base",
    "days_plot_s1_base", "days_plot_s2_base",
    "hired_labor_s1_base", "hired_labor_s2_base",
    "labor_cash_s1_base", "labor_cash_s2_base",
]

baseline_vars += [
    "non_agriculture_enterprise_avg_base",
    "electricity_access_avg_base",
]

static_covars = [
    "rain_mm_s1_2000_2010", "rain_mm_s2_2000_2010",
    "temp_s1_2000_2010", "temp_s2_2000_2010",
    "popdens_2010", "popdens_trend_2000_2010",
    "dist_major_town_km",
    "elev_mean_30km", "elev_sd_30km",
    "cropland_10_share_2010", "cropland_20_share_2010",
    "cropland_30_share_2010", "cropland_40_share_2010",
    "poverty_mean_2010", "poverty_sd_2010",
]

panel_wave = pd.concat([csb, ctrl], ignore_index=True)

panel_wave["Wave"] = num(panel_wave["Wave"]).astype("Int64")
panel_wave["treated"] = num(panel_wave["treated"]).astype("Int64")
panel_wave["latest_baseline_wave"] = num(panel_wave["latest_baseline_wave"]).astype("Int64")

panel_wave["is_baseline"] = (
    panel_wave["treated"].eq(0)
    | (
        panel_wave["treated"].eq(1)
        & panel_wave["latest_baseline_wave"].notna()
        & panel_wave["Wave"].le(panel_wave["latest_baseline_wave"])
    )
)

baseline_audit = (
    panel_wave
    .query("treated == 1")
    .groupby(["site_name", "treat_year", "latest_baseline_wave"], dropna=False)
    .agg(
        all_waves=("Wave", lambda s: ",".join(map(str, sorted(s.dropna().astype(int).unique())))),
        baseline_waves=("Wave", lambda s: ",".join(
            map(str, sorted(panel_wave.loc[s.index].query("is_baseline")["Wave"].dropna().astype(int).unique()))
        )),
    )
    .reset_index()
)

display(baseline_audit)

base = panel_wave[panel_wave["is_baseline"]].copy()

for col in baseline_vars + static_covars:
    if col in base.columns:
        base[col] = num(base[col])

def collapse_site(g):
    treat_year_vals = num(g["treat_year"]).dropna()
    latest_wave_vals = num(g["latest_baseline_wave"]).dropna()

    out = {
        "site_id": g.name,
        "site_name": g["site_name"].iloc[0],
        "treated": int(g["treated"].iloc[0]),
        "treat_year": treat_year_vals.iloc[0] if len(treat_year_vals) else pd.NA,
        "latest_baseline_wave": latest_wave_vals.iloc[0] if len(latest_wave_vals) else pd.NA,
        "baseline_waves": ",".join(map(str, sorted(g["Wave"].dropna().astype(int).unique()))),
        "n_site_wave_rows": len(g),
        "n_hh_baseline_total": num(g["n_HHID_wave"]).sum(min_count=1),
    }

    for col in baseline_vars:
        if col in g.columns:
            out[col] = weighted_mean(g[col], g["n_HHID_wave"])

    for col in static_covars:
        if col in g.columns:
            vals = num(g[col]).dropna()
            out[col] = vals.iloc[0] if len(vals) else pd.NA

    return pd.Series(out)


balance_panel = (
    base
    .set_index("site_id", drop=False)
    .groupby(level=0, group_keys=False)
    .apply(collapse_site)
    .reset_index(drop=True)
)

display(balance_panel.head())



,site_name,treat_year,latest_baseline_wave,all_waves,baseline_waves
0,ADIE CSB,2019,7,"1,2,3,4,5,7,8","1,2,3,4,5,7"
1,APAC CSB,2021,8,"1,2,3,4,5,7,8","1,2,3,4,5,7,8"
2,AWEI CSB,2024,8,"1,2,3,4,5,7,8","1,2,3,4,5,7,8"
3,Bunyoro Kitara CSB,2021,8,"1,2,3,4,5,7,8","1,2,3,4,5,7,8"
4,HOIMA CSB,2018,7,"1,2,3,4,5,7,8","1,2,3,4,5,7"
5,JOY M.CSB,2016,5,"1,2,3,4,5,7,8","1,2,3,4,5"
6,KIZIMBA CSB,2010,1,"1,2,3,4,5,7,8",1
7,NAKASEKE CSB,2014,4,"1,2,3,4,5,7,8","1,2,3,4"
8,NAKASONGOLA CSB,2019,7,"1,2,3,4,5,7,8","1,2,3,4,5,7"
9,NEM ANYIM CSB,2018,7,"1,2,3,4,5,7,8","1,2,3,4,5,7"


,site_id,site_name,treated,treat_year,latest_baseline_wave,baseline_waves,n_site_wave_rows,n_hh_baseline_total,share_hh_working_s1_base,share_hh_working_s2_base,...,popdens_trend_2000_2010,dist_major_town_km,elev_mean_30km,elev_sd_30km,cropland_10_share_2010,cropland_20_share_2010,cropland_30_share_2010,cropland_40_share_2010,poverty_mean_2010,poverty_sd_2010
0,CSB_ADIE CSB,ADIE CSB,1,2019,7,"1,2,3,4,5,7",6,350,0.604163,0.555203,...,0.027984,10.518076,1090.459946,30.184182,0.726212,0.137935,0.099451,0.012691,0.626265,0.048878
1,CSB_APAC CSB,APAC CSB,1,2021,8,"1,2,3,4,5,7,8",7,197,0.518114,0.562190,...,0.039622,18.600100,1109.750897,56.160473,0.720635,0.000000,0.061615,0.003140,0.557488,0.057309
2,CSB_AWEI CSB,AWEI CSB,1,2024,8,"1,2,3,4,5,7,8",7,123,0.549455,0.557788,...,0.024008,24.301528,1081.194236,26.651928,0.639608,0.000341,0.126591,0.019279,0.567222,0.069853
3,CSB_BUNYORO KITARA CSB,Bunyoro Kitara CSB,1,2021,8,"1,2,3,4,5,7,8",7,23,0.640972,0.687636,...,0.054940,9.306533,1015.151449,63.746500,0.247674,0.000511,0.231385,0.037281,0.743123,0.087822
4,CSB_HOIMA CSB,HOIMA CSB,1,2018,7,"1,2,3,4,5,7",6,201,0.642905,0.607662,...,0.035108,13.079227,1083.971694,36.581865,0.686432,0.000000,0.180653,0.070277,0.610345,0.045980


In [10]:
# Create balance table

balance_vars = [
    v for v in baseline_vars + static_covars
    if v in balance_panel.columns
]

def norm_diff(x_t, x_c):
    mt = x_t.mean(skipna=True)
    mc = x_c.mean(skipna=True)
    vt = x_t.var(skipna=True, ddof=1)
    vc = x_c.var(skipna=True, ddof=1)

    denom = np.sqrt((vt + vc) / 2)

    if pd.isna(denom) or denom == 0:
        return np.nan

    return (mt - mc) / denom

rows = []

for var in balance_vars:
    t = num(balance_panel.loc[balance_panel["treated"].eq(1), var])
    c = num(balance_panel.loc[balance_panel["treated"].eq(0), var])

    rows.append({
        "Variable": var,
        "Control Mean": c.mean(skipna=True),
        "CSB Mean": t.mean(skipna=True),
        "Norm. Diff.": norm_diff(t, c),
        "N Control": c.notna().sum(),
        "N CSB": t.notna().sum(),
    })

balance_table = pd.DataFrame(rows)

display(balance_table)

,Variable,Control Mean,CSB Mean,Norm. Diff.,N Control,N CSB
0,share_hh_working_s1_base,0.821194,0.577396,-0.226427,5172,17
1,share_hh_working_s2_base,0.569090,0.653183,0.540655,5177,17
2,avg_subsistence_base,0.186367,0.155168,-0.326787,5178,17
3,ag_extension_base,0.099410,0.177258,1.051782,5178,17
4,road_type_A_base,0.095197,0.279715,0.986711,5110,17
5,road_type_B_base,0.232659,0.230750,-0.011342,5110,17
6,road_type_C_base,0.777338,0.680820,-0.503418,5110,17
7,road_type_D_base,0.936799,0.825081,-1.106383,5110,17
8,time_to_road_A_base,20.630096,18.904224,-0.102988,4329,14
9,time_to_road_B_base,19.840161,22.700294,0.158858,4998,17


In [11]:
# Stata-safe type cleanup

stata_numeric_cols = [
    "treated",
    "treat_year",
    "latest_baseline_wave",
    "n_site_wave_rows",
    "n_hh_baseline_total",
]

for col in stata_numeric_cols:
    if col in balance_panel.columns:
        balance_panel[col] = pd.to_numeric(balance_panel[col], errors="coerce")

stata_string_cols = [
    "site_id",
    "site_name",
    "baseline_waves",
]

for col in stata_string_cols:
    if col in balance_panel.columns:
        balance_panel[col] = (
            balance_panel[col]
            .astype("string")
            .where(balance_panel[col].notna(), None)
        )

for col in ["Control Mean", "CSB Mean", "Norm. Diff.", "N Control", "N CSB"]:
    if col in balance_table.columns:
        balance_table[col] = pd.to_numeric(balance_table[col], errors="coerce")

if "Variable" in balance_table.columns:
    balance_table["Variable"] = (
        balance_table["Variable"]
        .astype("string")
        .where(balance_table["Variable"].notna(), None)
    )

In [12]:


OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

balance_panel.to_excel(OUT / "balance_panel_site_level_v2.xlsx", index=False)
balance_table.to_excel(OUT / "balance_table_unweighted_v2.xlsx", index=False)

balance_panel.to_csv(OUT / "balance_panel_site_level_v2.csv", index=False)
balance_table.to_csv(OUT / "balance_table_unweighted_v2.csv", index=False)

balance_panel.to_stata(OUT / "balance_panel_site_level_v2.dta", write_index=False, version=118)
balance_table.to_stata(OUT / "balance_table_unweighted_v2.dta", write_index=False, version=118)

print("Saved outputs to:", OUT)

Saved outputs to: C:\Users\Carl\Desktop\RA Data\Covariate balance\output


C:\Users\Carl\AppData\Local\Temp\ipykernel_25608\33453290.py:12: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    non_agriculture_enterprise_avg_base   ->   non_agriculture_enterprise_avg_b

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  balance_panel.to_stata(OUT / "balance_panel_site_level_v2.dta", write_index=False, version=118)
C:\Users\Carl\AppData\Local\Temp\ipykernel_25608\33453290.py:13: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    Control Mean   ->   Control_Mean
    CSB Mean   ->   CSB_Mean
    Norm. Diff.   ->   Norm__Diff_
    N Control   ->   N_Control
    N CSB   ->   N_CSB

If this is not what you expect, please make sure you have Stata-compliant
column nam